# EasyRAG Chunking Quality Analysis

## Chapter position

This chapter turns prepared documents into retrieval infrastructure. Chunk boundaries, embedding inputs, vector storage, and workspace artifacts all start here.

## Learning question

How do chunking choices show up later in retrieval quality instead of only changing chunk counts?

## Success criteria

- compare chunk counts, chunk sizes, and overlap metadata across strategies
- inspect how chunking choices affect later retrieval snippets
- identify chunking failure patterns before they turn into ranking problems

## Source code anchors

- `easyrag.rag.indexing.chunking.chunk_documents`
- `easyrag.rag.indexing.chunking_core.ChunkingConfig`
- `easyrag.rag.indexing.chunking_core.select_chunk_strategy`
- `easyrag.rag.orchestrator.EasyRAG.aquery`


## Method principles

- `easyrag.rag.indexing.chunking.chunk_documents`: This wrapper applies the chunking strategy registry across a document list. Its job is to turn document-level inputs into retrieval units while preserving the metadata needed later.
- `easyrag.rag.indexing.chunking_core.ChunkingConfig`: This config object holds the boundary policy for chunking. It controls window sizes, overlaps, section limits, and semantic split thresholds in one place.
- `easyrag.rag.indexing.chunking_core.select_chunk_strategy`: This selector chooses a default chunking policy from document hints such as file suffix or title. It is a routing decision, not a chunker by itself.
- `easyrag.rag.orchestrator.EasyRAG.aquery`: This is the public retrieval entry point. It delegates into the retrieval pipeline and returns a structured `QueryResult` rather than raw vector hits.


## How the code fits together

The flow in this notebook is same corpus -> multiple chunk strategies -> retrieval comparison. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.indexing import ChunkingConfig, chunk_documents
from easyrag.rag.indexing.pipeline import build_insert_payloads
from easyrag.support.optional_deps import Document

## What this cell is proving

We will run the same markdown-heavy document through several chunking setups and compare the resulting retrieval units side by side.

In [ ]:
analysis_document = Document(
    page_content=(
        "# Overview\nEasyRAG keeps repository guidance inspectable.\n"
        "## Retrieval\nQuery rewrite expands recall while hybrid retrieval keeps multiple signals visible.\n"
        "## Generation\nAnswer synthesis should stay tied to citations and packed context.\n"
        "## Evaluation\nMetrics localize whether the issue is retrieval, packing, or answer grounding.\n"
    ),
    metadata={
        "doc_id": "doc::analysis",
        "path": "docs/analysis.md",
        "relative_path": "docs/analysis.md",
        "title": "analysis",
        "source_type": "doc",
    },
)
rag = EasyRAG(
    working_dir=tempfile.mkdtemp(),
    workspace="chunk-quality-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)

configs = {
    "structured_default": ChunkingConfig(),
    "small_windows": ChunkingConfig(
        sliding_window_size=80,
        sliding_window_overlaps=(20, 30, 40),
        structured_max_section_chars=90,
    ),
    "semantic_friendly": ChunkingConfig(
        semantic_target_chars=80, semantic_similarity_threshold=0.9
    ),
}

summary = {}
for label, config in configs.items():
    chunks = chunk_documents([analysis_document], config=config, rag=rag)
    summary[label] = {
        "chunk_count": len(chunks),
        "strategies": sorted(
            {str(chunk.metadata.get("chunk_strategy")) for chunk in chunks}
        ),
        "chunk_lengths": [len(chunk.page_content) for chunk in chunks],
        "overlap_policies": [
            str(chunk.metadata.get("overlap_policy")) for chunk in chunks
        ],
        "previews": [
            chunk.page_content[:100].replace("\n", " ") for chunk in chunks[:4]
        ],
    }
_print_json(summary)

## Why this output looks like this

The next cell pushes two chunking policies into full workspaces so you can see the retrieval-side effect of the boundaries, not just the chunk counts.

In [ ]:
def run_chunk_policy(label: str, override: str | None):
    tmp = tempfile.TemporaryDirectory()
    rag = EasyRAG(
        working_dir=tmp.name,
        workspace=label,
        embedding_func=_stub_embedding,
        query_model_func=_stub_query_model,
    )
    run_sync(rag.initialize_storages())
    try:
        payload = build_insert_payloads(
            rag, [analysis_document], chunk_strategy_override=override
        )
        run_sync(rag.ainsert_documents([analysis_document]))
        result = run_sync(
            rag.aquery(
                "How does EasyRAG keep answer synthesis inspectable?",
                QueryParam(
                    mode="naive",
                    rewrite_enabled=False,
                    mqe_enabled=False,
                    chunk_top_k=3,
                ),
            )
        )
        return {
            "payload_chunk_count": len(payload["chunks"]),
            "citations": result.citations,
            "metadata": result.metadata,
        }
    finally:
        run_sync(rag.finalize_storages())
        tmp.cleanup()


structured_view = run_chunk_policy("structured-policy", "structured")
sliding_view = run_chunk_policy("sliding-policy", "sliding_window")
_print_json({"structured": structured_view, "sliding_window": sliding_view})

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

Provider-backed semantic chunking matters most when sentence embeddings are strong. The optional cell keeps the same document but lets the semantic chunker call the configured embedding function if available.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_rag = EasyRAG(workspace="provider-chunk-quality-demo")
    try:
        provider_chunks = chunk_documents(
            [analysis_document],
            config=ChunkingConfig(semantic_target_chars=80),
            rag=provider_rag,
        )
        _print_json([chunk.metadata for chunk in provider_chunks])
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")

## Common mistakes / debugging cues

- Do not tune retrieval before you understand chunk boundaries, embedding inputs, and backend choice.
- Watch metadata such as `chunk_strategy`, `vector_backend`, and workspace artifacts, not only final query hits.
- Indexing bugs often look like retrieval bugs until you inspect the payloads that were actually written.

## Takeaway

A chunking policy changes three things at once: how many retrieval units exist, how much context each unit carries, and how much overlap later ranking has to sort through. That is why chunking quality should be inspected before you decide a retriever is weak.

## Where to go next

- Continue with [03_03_embeddings_basics.ipynb](03_03_embeddings_basics.ipynb) to follow the chunking output into embedding space.
- Read [03-indexing-overview.md](../../docs/03-indexing-overview.md) for the indexing-stage map.
- Revisit [03_01_chunking_principles.ipynb](03_01_chunking_principles.ipynb) if you want the strategy-level walkthrough before this quality comparison.